# ローカル LLM 実践 Notebook 目次

このフォルダは `docs/local-llm-customization` の実践編ですが、Notebook 単体でも学べるように、各章に目的、手順、コード、結果の読み方を入れています。docs は背景を詳しく読み返すための補助線です。

第1章から第4章では、Ollama を呼び出し、Chat UI、Python API、RAG、agentic coding、レビューの入口を小さく体験します。第5章では事前学習済みモデルに LoRA adapter を実際に学習します。第6章では同じモデルに対して、短い教材コーパスで continued-pretraining 風の追加学習を実行します。

| 章 | Notebook | 実践すること |
|---:|---|---|
| 1 | [01-overview.ipynb](01-overview.ipynb) | Ollama 確認、Python API、Chat UI、aider の安全な入口を体験する |
| 2 | [02-prompt-design.ipynb](02-prompt-design.ipynb) | Chat UI と Python API で短い依頼と構造化依頼を比較する |
| 3 | [03-rag.ipynb](03-rag.ipynb) | docs を検索し、根拠なし回答と根拠つき回答を比べる |
| 4 | [04-tool-use.ipynb](04-tool-use.ipynb) | Chat UI、API、coding agent、レビューの使い分けを試す |
| 5 | [05-lora-qlora.ipynb](05-lora-qlora.ipynb) | 事前学習済みモデルに LoRA を付け、教材データで実学習する |
| 6 | [06-continued-pretraining.ipynb](06-continued-pretraining.ipynb) | 教材コーパスで continued-pretraining 風の追加学習を実行する |
| 7 | [07-evaluation-operation.ipynb](07-evaluation-operation.ipynb) | 評価ケースを実行し、運用チェックへつなげる |


In [ ]:
from pathlib import Path
import os
import json
import sys

# Notebook をどこから開いても helper を import できるようにします。
search_roots = [Path.cwd()]
env_root = os.environ.get("LOCAL_LLM_REPO_ROOT")
if env_root:
    search_roots.append(Path(env_root))
search_roots.append(Path("C:/LLM"))

seen = set()
for root in search_roots:
    current = root.resolve()
    for candidate in [current, *current.parents]:
        if candidate in seen:
            continue
        seen.add(candidate)
        helper_dir = candidate / "notebooks"
        if (helper_dir / "local_llm_practice.py").exists():
            sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise RuntimeError(
        "notebooks/local_llm_practice.py が見つかりません。"
        "C:/LLM か notebooks/ 配下で開くか、LOCAL_LLM_REPO_ROOT を設定してください。"
    )

from local_llm_practice import (
    DATA_DIR,
    DOCS_DIR,
    REPO_ROOT,
    WORK_DIR,
    TrainingConfig,
    ask_about_chapter,
    attach_lora,
    configure_local_caches,
    count_trainable_parameters,
    generate_text,
    gpu_summary,
    load_base_model,
    load_chapter,
    load_jsonl,
    make_cpt_features,
    make_sft_features,
    nvidia_smi_summary,
    ollama_generate,
    print_headings,
    read_text,
    retrieve_chunks,
    split_markdown,
    train_lora_adapter,
)

configure_local_caches()
print("REPO_ROOT:", REPO_ROOT)
print("DOCS_DIR :", DOCS_DIR)
print("WORK_DIR :", WORK_DIR)

# 章ファイルと Notebook がそろっているか確認します。
chapters = [
    ("01-overview.md", "01-overview.ipynb"),
    ("02-prompt-design.md", "02-prompt-design.ipynb"),
    ("03-rag.md", "03-rag.ipynb"),
    ("04-tool-use.md", "04-tool-use.ipynb"),
    ("05-lora-qlora.md", "05-lora-qlora.ipynb"),
    ("06-continued-pretraining.md", "06-continued-pretraining.ipynb"),
    ("07-evaluation-operation.md", "07-evaluation-operation.ipynb"),
]
for doc_name, nb_name in chapters:
    print(f"{doc_name:34} {'OK' if (DOCS_DIR / doc_name).exists() else 'missing'}")
    print(f"{nb_name:34} {'OK' if (REPO_ROOT / 'notebooks' / 'local-llm-customization' / nb_name).exists() else 'missing'}")


## 学習成果物の扱い

第5章と第6章の adapter、キャッシュ、検証ログは `work/` 配下に保存します。`work/` は `.gitignore` されているため、学習済み重みやローカル環境の状態を公開 repository に混ぜません。
